# Gizli Dirichlet Dağılımı (Latent Dirichlet Allocation) (LDA)

🎯 Bu challenge'ın amacı, **LDA** algoritması (NLP'de Denetimsiz Öğrenme) ile e-posta külliyatı içinde konular bulmaktır.

✉️ İşte 1000'den fazla ***etiketlenmemiş e-posta*** içeren bir koleksiyon. Bunlardan ***konuları çıkarmaya*** çalışalım!

In [1]:
import pandas as pd

url = 'https://d32aokrjazspmn.cloudfront.net/materials/lda_data'

data = pd.read_csv(url, sep=",", header=None)
data.columns = ['text']
data.head()

,text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...


In [5]:
data.loc[0, "text"]

'From: gld@cunixb.cc.columbia.edu (Gary L Dare)\nSubject: Stan Fischler, 4/4\nSummary: From the Devils pregame show, prior to hosting the Penguins\nNntp-Posting-Host: cunixb.cc.columbia.edu\nReply-To: gld@cunixb.cc.columbia.edu (Gary L Dare)\nOrganization: PhDs In The Hall\nLines: 32\n\n\nAt the Lester Patrick Awards lunch, Bill Torrey mentioned that one of his\noptions next season is to be president of the Miami team, with Bob Clarke\nworking for him.  At the same dinner, Clarke said that his worst mistake\nin Philadelphia was letting Mike Keenan go -- in retrospect, almost all\nplayers came realize that Keenan knew what it took to win.  Rumours are\nnow circulating that Keenan will be back with the Flyers.\n\nNick Polano is sick of being a scapegoat for the schedule made for the\nRed Wings; After all, Bryan Murray approved it.\n\nGerry Meehan and John Muckler are worried over the Sabres\' prospects;\nAssistant Don Lever says that the Sabres have to get their share now,\nbecause a Que

In [2]:
data.shape

(1199, 1)

## (1) Preprocessing 

❓ **Question (Cleaning**) ❓ You're used to it by now... Clean up! Store the cleaned text in a new column "clean_text" of the DataFrame.

In [29]:
# SENİN KODUN BURAYA
import string
from nltk.corpus import stopwords
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer

def preprocessing(sentence):
    # remove blank spaces
    sentence = sentence.strip()
    # remove \n
    sentence = sentence.replace('\n', ' ')
    # reverte to lowercase
    sentence = sentence.lower()
    # remove numbers
    sentence = sentence.translate(str.maketrans('', '', string.digits))
    # remove punctuation
    punctuation = string.punctuation.replace("@", "")
    sentence = sentence.translate(str.maketrans('', '', punctuation))
    sentence = sentence.replace(" @ ", "@")
    # tokenize the sentence
    tokens = word_tokenize(sentence)
    # remove stop words    
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    # lemmatize the tokens
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return ' '.join(tokens)
data["clean_text"] = data["text"].apply(preprocessing)
data.head()

,text,clean_text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...,gld @ cunixbcccolumbiaedu gary l dare subject ...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlep @ velaacsoaklandedu cardinal ximenez ...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...,miner @ kuhubccukansedu subject ancient book o...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlep @ velaacsoaklandedu cardinal ximenez ...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...,vzhivov @ superiorcarletonca vladimir zhivov s...


In [30]:
data.loc[0, "clean_text"]

'gld @ cunixbcccolumbiaedu gary l dare subject stan fischler summary devil pregame show prior hosting penguin nntppostinghost cunixbcccolumbiaedu replyto gld @ cunixbcccolumbiaedu gary l dare organization phd hall line lester patrick award lunch bill torrey mentioned one option next season president miami team bob clarke working dinner clarke said worst mistake philadelphia letting mike keenan go retrospect almost player came realize keenan knew took win rumour circulating keenan back flyer nick polano sick scapegoat schedule made red wing bryan murray approved gerry meehan john muckler worried sabre prospect assistant lever say sabre get share quebec dynasty emerging mighty duck declared throw money around loosely buy team oiler coach ted green remarked guy around fill tie domis skate none fill helmet senator andrew mcbain told security guard chicago stadium warned stair leading locker room mcbain mouthed seasoned professional tumbled entire steep flight gld je souviens gary l dare gl

## (2) Latent Dirichlet Allocation model

❓ **Soru (Eğitim)** ❓ Potansiyel konuları çıkarmak için bir LDA modeli eğitin

In [31]:
# SENİN KODUN BURAYA
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(data["clean_text"])

model = LatentDirichletAllocation(n_components=5, random_state=42)
model.fit(X)

LatentDirichletAllocation(n_components=5, random_state=42)

##  (3) Potansiyel konuları görselleştirin

🎁 Potansiyel konularla ilişkili kelimeleri yazdırmak için bir  fonksiyon kodladık.

In [32]:
def print_topics(model, vectorizer):
    for idx, topic in enumerate(model.components_):
        print("Topic %d:" % (idx))
        print([(vectorizer.get_feature_names_out()[i], topic[i])
                        for i in topic.argsort()[:-10 - 1:-1]])

❓ **Soru** ❓ LDA tarafından çıkarılan konuları yazdırın.

In [33]:
# SENİN KODUN BURAYA
print_topics(model, vectorizer)

Topic 0:
[('hell', 214.90733722214452), ('would', 161.8621683045541), ('subject', 143.87575165534957), ('absolute', 137.36523656667094), ('line', 136.394384952545), ('writes', 126.42605434590052), ('organization', 122.81974337510822), ('god', 113.58276857493654), ('think', 112.45182420090596), ('university', 102.71474980823751)]
Topic 1:
[('gm', 154.1103341073033), ('bos', 124.57409336960177), ('det', 116.13342695338274), ('chi', 112.82281684840532), ('subject', 104.04892819432791), ('organization', 103.6108669470755), ('line', 102.49750238948653), ('tor', 102.18726602111876), ('van', 99.1906635884968), ('pit', 94.57858536710143)]
Topic 2:
[('pt', 302.3908729721967), ('period', 230.02359987739638), ('la', 155.77174424410524), ('power', 153.30102491017482), ('play', 136.3857690103545), ('pp', 132.26255023681497), ('flyer', 106.15941716887846), ('captain', 97.29233043601793), ('second', 88.73469077022075), ('shot', 86.82236381964427)]
Topic 3:
[('game', 882.7631675199476), ('team', 849.3

## (4) Yeni bir metnin belge-konu karışımını tahmin edin

❓ **Soru (Tahmin)** ❓

LDA modeliniz fit edildiğine göre, onu yeni bir metnin konularını tahmin etmek için kullanabilirsiniz.

1. Örneği vektörleştirin
2. Vektörleştirilmiş örnek üzerinde LDA'yı kullanarak konuları tahmin edin

In [34]:
example = ["My team performed poorly last season. Their best player was out injured and only played one game"]

In [35]:
# SENİN KODUN BURAYA
example = ["My team performed poorly last season. Their best player was out injured and only played one game"]

processed_example = preprocessing(example[0])

example_vectorized = vectorizer.transform([processed_example])  # ❗ list içine aldık
example_distribution = model.transform(example_vectorized)

print(example_distribution)

[[0.01684371 0.01671328 0.01670937 0.93290965 0.01682399]]


🏁 Tebrikler! LDA'yı hızlı bir şekilde nasıl uygulayacağınızı öğrendiniz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

🚀 ... ve bir sonraki göreve geçin!